In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# =========================================
# START DRIVER
# =========================================
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# =========================================
# OPEN SITE
# =========================================
driver.get("https://www.wacdl.org/AF_MemberDirectory.asp")

print("1. Select Category")
print("2. Click Submit")
print("3. Wait until attorney list appears")

input("Press ENTER when results page is fully loaded and you see Page 1...")

# =========================================
# ALL RECORDS
# =========================================
all_records = []
page_number = 1

# Free domains check list
free_domains = ["gmail", "yahoo", "hotmail", "outlook", "icloud", "aol", "msn", "live"]

# =========================================
# PAGINATION LOOP
# =========================================
while True:
    print(f"\n========== PROCESSING PAGE {page_number} ==========")
    
    # 5 seconds wait taake naya page completely load ho jaye
    time.sleep(5) 

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # =====================================
    # FIND ATTORNEY BLOCKS
    # =====================================
    blocks = soup.find_all("td")
    filtered_blocks = []

    for block in blocks:
        text = block.get_text(" ", strip=True)
        if ("Phone" in text or "Email" in text or "Website" in text) and "WA" in text:
            filtered_blocks.append(block)

    print(f"Found {len(filtered_blocks)} attorney blocks on Page {page_number}")

    # =====================================
    # PROCESS EACH BLOCK
    # =====================================
    for block in filtered_blocks:
        try:
            section_text = block.get_text("\n", strip=True)
            lines = [x.strip() for x in section_text.split("\n") if x.strip()]

            if not lines:
                continue

            # NAME
            name = lines[0]

            # ADDRESS COMPONENTS
            address = ""
            city = ""
            state = ""
            zipcode = ""

            for i, line in enumerate(lines):
                if re.search(r'\bWA\b\s*\d{5}', line):
                    match = re.search(r'^(.*?),\s*([A-Z]{2})\s*(\d{5}(?:-\d{4})?)', line)
                    if match:
                        city = match.group(1).strip()
                        state = match.group(2).strip()
                        zipcode = match.group(3).strip()

                    if i > 0:
                        address = lines[i - 1]
                    break

            # PHONE
            phone = ""
            phone_match = re.search(r'Phone:\s*(.*)', section_text)
            if phone_match:
                phone = phone_match.group(1).strip()

            # EMAIL
            email = ""
            email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', section_text)
            if email_match:
                email = email_match.group(0)

            # SKIP .EDU
            if email.endswith(".edu"):
                continue

            # WEBSITE
            website = ""
            web_match = re.search(r'(http[s]?://[^\s]+)', section_text)
            if web_match:
                website = web_match.group(1)

            # COMPANY DETECTION
            company = ""
            if website:
                domain = re.sub(r'https?://(www\.)?', '', website).split('/')[0]
                base_name = domain.split(".")[0]
                base_name = re.sub(r'[-_]', ' ', base_name)
                company = base_name.upper()
            elif email:
                domain = email.split("@")[-1]
                base_name = domain.split(".")[0].lower()
                if base_name not in free_domains:
                    base_name = re.sub(r'[-_]', ' ', base_name)
                    company = base_name.upper()

            if not company:
                for line in lines:
                    lower_line = line.lower()
                    if any(word in lower_line for word in ["law", "firm", "legal", "attorneys", "office", "group"]):
                        company = line.strip().upper()
                        break

            # SKIP IF NO ADDRESS
            if not address:
                continue

            # APPEND TO MASTER LIST
            all_records.append({
                "Recipient": name,
                "Company": company,
                "Address": address,
                "City": city,
                "State": state,
                "Zip Code": zipcode,
                "Phone Number": phone,
                "Email": email,
                "Website": website
            })

        except Exception as e:
            print(f"Error parsing block: {e}")

    # =====================================
    # AUTOMATIC NEXT PAGE CLICK (UPDATED)
    # =====================================
    try:
        # Link text ">" ko dhoonde ga jo agle page par le jata hai
        next_button = driver.find_element(By.LINK_TEXT, ">")
        
        print("Clicking '>' arrow button for the next page...")
        driver.execute_script("arguments[0].click();", next_button)
        
        page_number += 1
        
    except Exception:
        print("\nLast page reached or '>' button not clickable. Loop completed.")
        break

# =========================================
# CREATE DATAFRAME & CLEAN DUPLICATES
# =========================================
if all_records:
    df = pd.DataFrame(all_records)

    # Combined deduplication
    df = df.drop_duplicates(subset=["Recipient"])
    df = df.drop_duplicates(subset=["Address"])
    df = df[df["Address"].astype(str).str.strip() != ""]

    # =========================================
    # EXPORT TO EXCEL
    # =========================================
    output_path = r"xyz"
    
    df.to_excel(output_path, index=False)

    print("\n=================================")
    print("SUCCESS")
    print(f"Total Unique Rows Collected: {len(df)}")
    print(f"Excel Saved Successfully at: {output_path}")
    print("=================================")
else:
    print("\nKoi data collect nahi ho saka.")

driver.quit()